# 09 · Trees and Binary Search Trees

After this notebook you can build a binary tree from a LeetCode-style list, draw it, traverse it four ways (recursively and iteratively), and solve the classic tree and BST questions — depth, balance, invert, diameter, validate, lowest common ancestor, serialize, k-th smallest — with two templates.

**Time:** ~40 min · **Runs:** offline on CPU in < 1 min · **Needs:** [Recursion and backtracking](08_recursion_and_backtracking.ipynb), [Stacks and queues](04_stacks_and_queues.ipynb)

## Why this matters in interviews

- Trees are the most common "medium" topic, and almost every tree question is one of two templates: **DFS that returns a value to its parent** or **BFS level by level**. Say which one you are using and why.
- The BST property (left < node < right) makes search O(height). The follow-up is always: *what is the height?* `log n` if balanced, `n` if you inserted sorted data.
- Interviewers love the traps here: validating a BST by only checking children, forgetting that the diameter need not pass through the root, recursion depth on a skewed tree, and serialising without null markers.
- Trees are everywhere in AI engineering code too: JSON payloads, document heading outlines used for chunking, parse trees, decision trees.

## What interviewers ask

- Traversals: pre-order, in-order, post-order, level order — recursive **and** iterative (LeetCode 144 / 94 / 145 / 102).
- Maximum depth (104), balanced tree (110), invert tree (226), diameter (543).
- BST: search and insert (700 / 701), validate (98), k-th smallest (230), lowest common ancestor (235).
- Lowest common ancestor in a general binary tree (236).
- Serialize and deserialize a binary tree (297).
- Follow-ups: "iterative please", "what if the tree is a million nodes deep?", "what if the BST is modified often and you need k-th smallest repeatedly?"

In [ ]:
import random
from collections import deque

import matplotlib.pyplot as plt

plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e5e0", "axes.axisbelow": True, "axes.titlesize": 11,
                     "legend.frameon": False})
BLUE, ORANGE, AQUA, INK, MUTED = "#2a78d6", "#eb6834", "#1baf7a", "#0b0b0b", "#52514e"

class TreeNode:
    __slots__ = ("val", "left", "right")
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right
    def __repr__(self):
        return f"TreeNode({self.val})"

## 1. The pattern

**In plain English:** a binary tree is a node that holds a value and up to two children, each of which is again a tree (or nothing). Because the definition is recursive, the code is recursive: solve it for the left child, solve it for the right child, combine. Every tree problem comes down to *"what do I need from my children, and what do I hand to my parent?"*

**Template 1 — DFS that returns a value (post-order):**
```python
def solve(node):
    if node is None:
        return BASE                 # the answer for an empty tree
    left = solve(node.left)
    right = solve(node.right)
    return combine(node.val, left, right)
```
**Template 2 — BFS, one level at a time:**
```python
queue = deque([root])
while queue:
    for _ in range(len(queue)):     # exactly the nodes of the current level
        node = queue.popleft()
        ...                         # visit, then enqueue node.left / node.right
```
Recognise it: "depth / height / balanced / path / subtree / ancestor" → Template 1. "level / row / nearest / minimum depth / right side view" → Template 2. Both are O(n) time; DFS uses O(h) stack space, BFS O(width) queue space.

### Build a tree from a LeetCode list

LeetCode writes trees level by level with `None` for a missing child: `[3, 9, 20, None, None, 15, 7]`. `build_tree` reads that format with a queue; `to_level_list` writes it back, so we can assert the round trip.

In [ ]:
def build_tree(values):
    """LeetCode level-order list -> root TreeNode."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue, i = deque([root]), 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i]); queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i]); queue.append(node.right)
        i += 1
    return root

def to_level_list(root):
    """Inverse of build_tree (trailing Nones trimmed)."""
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        out.append(None if node is None else node.val)
        if node is not None:
            queue.extend([node.left, node.right])
    while out and out[-1] is None:
        out.pop()
    return out

for values in ([3, 9, 20, None, None, 15, 7], [1, None, 2, 3], [], [1], [5, 4, 6, None, None, 3, 7]):
    assert to_level_list(build_tree(values)) == values
print("round trip OK:", to_level_list(build_tree([3, 9, 20, None, None, 15, 7])))

### A drawing helper

Position each node at x = its in-order index and y = −depth. That layout never overlaps, and for a BST it has a bonus: the values increase left to right, so you can *see* the BST property.

In [ ]:
def tree_positions(root):
    pos, x = {}, 0
    def walk(node, depth):
        nonlocal x
        if node is None:
            return
        walk(node.left, depth + 1)
        pos[node] = (x, -depth); x += 1
        walk(node.right, depth + 1)
    walk(root, 0)
    return pos

def draw_tree(root, ax=None, title=None, highlight=(), notes=None):
    """highlight: nodes to colour orange; notes: {node: small label} drawn above-right."""
    own = ax is None
    if own:
        fig, ax = plt.subplots(figsize=(5.5, 2.8))
    pos = tree_positions(root)
    for node, (x, y) in pos.items():
        for child in (node.left, node.right):
            if child is not None:
                ax.plot([x, pos[child][0]], [y, pos[child][1]], color="#b5b3ac", lw=1.5, zorder=1)
    for node, (x, y) in pos.items():
        hot = node in highlight
        ax.scatter([x], [y], s=480, zorder=2, linewidths=1.5,
                   color=ORANGE if hot else "#e7f0ff", edgecolors=ORANGE if hot else BLUE)
        ax.text(x, y, str(node.val), ha="center", va="center", fontsize=10, zorder=3,
                color="white" if hot else INK)
        if notes and node in notes:
            ax.text(x + 0.3, y + 0.25, str(notes[node]), fontsize=9, color=ORANGE, fontweight="bold")
    xs = [x for x, _ in pos.values()] or [0]
    ys = [y for _, y in pos.values()] or [0]
    ax.set_xlim(min(xs) - 0.8, max(xs) + 0.8); ax.set_ylim(min(ys) - 0.6, 0.6)
    ax.axis("off")
    if title:
        ax.set_title(title, fontsize=10)
    if own:
        plt.show()

example = build_tree([3, 9, 20, None, None, 15, 7])
bst = build_tree([8, 3, 10, 1, 6, None, 14, None, None, 4, 7, 13])
fig, axes = plt.subplots(1, 2, figsize=(10, 2.8))
draw_tree(example, axes[0], "binary tree [3, 9, 20, None, None, 15, 7]")
draw_tree(bst, axes[1], "BST: values increase left to right")
plt.show()

## 2. Traversals: four orders, one tree

| Traversal | Order | Typical use |
|---|---|---|
| pre-order | node, left, right | copy / serialise a tree (parent before children) |
| in-order | left, node, right | a BST comes out **sorted** |
| post-order | left, right, node | compute from children up: heights, sizes, deleting a tree |
| level order (BFS) | level by level | anything about levels, nearest nodes, minimum depth |

Recursive versions append to one list (not `left + [val] + right`, which copies lists and turns O(n) into O(n²) on skewed trees). Iterative versions replace the call stack with your own: pre-order pops a node and pushes right then left; in-order walks left as far as possible, pops, then turns right; post-order is "pre-order with the children swapped, reversed".

In [ ]:
def preorder(root):
    out = []
    def walk(node):
        if node:
            out.append(node.val); walk(node.left); walk(node.right)
    walk(root)
    return out

def inorder(root):
    out = []
    def walk(node):
        if node:
            walk(node.left); out.append(node.val); walk(node.right)
    walk(root)
    return out

def postorder(root):
    out = []
    def walk(node):
        if node:
            walk(node.left); walk(node.right); out.append(node.val)
    walk(root)
    return out

def level_order(root):
    levels, queue = [], deque([root] if root else [])
    while queue:
        level = []
        for _ in range(len(queue)):
            node = queue.popleft()
            level.append(node.val)
            queue.extend(child for child in (node.left, node.right) if child)
        levels.append(level)
    return levels

print("pre  :", preorder(bst))
print("in   :", inorder(bst), "<- sorted, because it is a BST")
print("post :", postorder(bst))
print("level:", level_order(bst))
assert inorder(bst) == sorted(inorder(bst))

In [ ]:
def preorder_iter(root):
    out, stack = [], [root] if root else []
    while stack:
        node = stack.pop()
        out.append(node.val)
        if node.right: stack.append(node.right)     # pushed first, popped last
        if node.left: stack.append(node.left)
    return out

def inorder_iter(root):
    out, stack, node = [], [], root
    while stack or node:
        while node:                                  # slide as far left as possible
            stack.append(node); node = node.left
        node = stack.pop()
        out.append(node.val)
        node = node.right                            # then do the right subtree
    return out

def postorder_iter(root):
    out, stack = [], [root] if root else []
    while stack:                                     # visits node, right, left ...
        node = stack.pop()
        out.append(node.val)
        if node.left: stack.append(node.left)
        if node.right: stack.append(node.right)
    return out[::-1]                                 # ... reversed: left, right, node

def random_tree(n, seed):
    rnd = random.Random(seed)
    return build_tree([rnd.randint(0, 99)] + [rnd.randint(0, 99) if rnd.random() < 0.7 else None for _ in range(n)])

for seed in range(200):
    t = random_tree(seed % 40, seed)
    assert preorder_iter(t) == preorder(t) and inorder_iter(t) == inorder(t) and postorder_iter(t) == postorder(t)
assert preorder(None) == inorder_iter(None) == postorder_iter(None) == [] and level_order(None) == []
print("iterative == recursive on 200 random trees (and the empty tree)")

The same tree, with the visit order written next to each node — this is the picture to hold in your head:

In [ ]:
def visit_numbers(root, order_fn):
    nodes = {}
    def index(node):                     # map value -> node (values are unique here)
        if node:
            nodes[node.val] = node; index(node.left); index(node.right)
    index(root)
    return {nodes[v]: i + 1 for i, v in enumerate(order_fn(root))}

orders = [("pre-order", preorder), ("in-order", inorder), ("post-order", postorder),
          ("level order", lambda r: [v for level in level_order(r) for v in level])]
fig, axes = plt.subplots(1, 4, figsize=(11, 2.8))
for ax, (name, fn) in zip(axes, orders):
    draw_tree(example, ax, name, notes=visit_numbers(example, fn))
fig.suptitle("Visit order (orange numbers) of the four traversals on [3, 9, 20, None, None, 15, 7]")
plt.tight_layout(); plt.show()

**Why iterative matters:** recursion depth equals tree height. A tree built from sorted data is a 5,000-long chain; the recursive in-order dies, the iterative one does not.

In [ ]:
chain = TreeNode(0)
node = chain
for v in range(1, 5000):                 # a right-leaning chain: height 5000
    node.right = TreeNode(v); node = node.right

try:
    inorder(chain)
    raise AssertionError("expected RecursionError")
except RecursionError:
    print("recursive in-order on a 5,000-deep chain -> RecursionError")
assert inorder_iter(chain) == list(range(5000))
print("iterative in-order -> 5,000 values, fine")

## 3. Classic tree problems

### Problem 1 — Maximum depth (LeetCode 104)

Number of nodes on the longest root-to-leaf path. `[3, 9, 20, None, None, 15, 7]` → 3. Template 1 directly: `1 + max(left, right)`; or Template 2: count the levels.

In [ ]:
def max_depth(root):
    return 0 if root is None else 1 + max(max_depth(root.left), max_depth(root.right))

def max_depth_bfs(root):
    return len(level_order(root))

assert max_depth(example) == max_depth_bfs(example) == 3
assert max_depth(None) == 0 and max_depth(TreeNode(1)) == 1 and max_depth_bfs(chain) == 5000
for seed in range(100):
    t = random_tree(30, seed)
    assert max_depth(t) == max_depth_bfs(t)
print("max depth of the example tree:", max_depth(example))

**Complexity:** O(n) time; O(h) space for DFS, O(width) for BFS.

### Problem 2 — Balanced binary tree (LeetCode 110)

Balanced = at every node the left and right heights differ by at most 1.

**Brute force:** at every node call `height()` on both children, then recurse — heights get recomputed at every ancestor: O(n log n) on balanced trees and O(n²) in the worst case. **Optimal:** one post-order pass that returns the height, or `-1` as a sentinel meaning "already unbalanced, stop".

In [ ]:
calls = 0
def height(node):
    global calls
    calls += 1
    return 0 if node is None else 1 + max(height(node.left), height(node.right))

def is_balanced_naive(root):
    if root is None:
        return True
    return (abs(height(root.left) - height(root.right)) <= 1
            and is_balanced_naive(root.left) and is_balanced_naive(root.right))

def is_balanced(root):
    def check(node):                     # height of node, or -1 if some subtree is unbalanced
        global calls
        calls += 1
        if node is None:
            return 0
        lh = check(node.left)
        if lh == -1:
            return -1
        rh = check(node.right)
        if rh == -1 or abs(lh - rh) > 1:
            return -1
        return 1 + max(lh, rh)
    return check(root) != -1

perfect = build_tree(list(range(1, 2 ** 11)))        # 2047 nodes, perfectly balanced
call_counts = {}
for name, fn in [("naive", is_balanced_naive), ("one pass", is_balanced)]:
    calls = 0
    assert fn(perfect) is True
    call_counts[name] = calls
    print(f"{name:>8}: balanced=True  calls={calls:,}")
assert call_counts["naive"] > 5 * call_counts["one pass"]
assert not is_balanced(build_tree([1, 2, 2, 3, 3, None, None, 4, 4])) and is_balanced(None)
for seed in range(200):
    t = random_tree(25, seed)
    assert is_balanced(t) == is_balanced_naive(t)

**Complexity:** one pass is O(n) time and O(h) space. The call counts show the naive version doing roughly `log n` times more work even on a perfectly balanced tree.

### Problem 3 — Invert a binary tree (LeetCode 226)

Mirror the tree: swap every node's children. One line of recursion (any traversal order works, as long as every node is swapped once).

In [ ]:
def invert(root):
    if root:
        root.left, root.right = invert(root.right), invert(root.left)
    return root

t = build_tree([4, 2, 7, 1, 3, 6, 9])
before = level_order(t)
fig, axes = plt.subplots(1, 2, figsize=(10, 2.6))
draw_tree(t, axes[0], "before")
invert(t)
draw_tree(t, axes[1], "after invert: every level reads backwards")
plt.show()
assert level_order(t) == [level[::-1] for level in before]
assert invert(None) is None

**Complexity:** O(n) time, O(h) space.

### Problem 4 — Diameter (LeetCode 543)

The number of **edges** on the longest path between any two nodes. The path does **not** have to pass through the root — that is the trap.

**Brute force:** at every node, `height(left) + height(right)` is the longest path bending there; take the max over all nodes, recomputing heights each time: O(n²). **Optimal:** the same idea inside a single height DFS that updates a running best.

In [ ]:
def diameter_naive(root):
    if root is None:
        return 0
    through_here = height(root.left) + height(root.right)
    return max(through_here, diameter_naive(root.left), diameter_naive(root.right))

def diameter(root):
    best = 0
    def h(node):
        nonlocal best
        if node is None:
            return 0
        lh, rh = h(node.left), h(node.right)
        best = max(best, lh + rh)            # longest path that bends at this node
        return 1 + max(lh, rh)
    h(root)
    return best

assert diameter(build_tree([1, 2, 3, 4, 5])) == 3
off_root = build_tree([1, 2, None, 3, 4, 5, None, None, 6])   # longest path 5-3-2-4-6 skips the root
through_root = height(off_root.left) + height(off_root.right)
assert diameter(off_root) == 4 and through_root == 3
assert diameter(None) == 0 and diameter(TreeNode(1)) == 0
for seed in range(200):
    t = random_tree(30, seed)
    assert diameter(t) == diameter_naive(t)
print(f"off-root example: diameter {diameter(off_root)}, best path through the root only {through_root}")

**Complexity:** O(n) time, O(h) space.

## 4. Binary search trees

**In plain English:** a BST keeps everything smaller on the left and everything larger on the right — for *every* node, not just its children. That lets search throw away half the remaining tree at each step, like binary search, so search/insert/delete cost O(h).

### Problem 5 — Insert, search, validate (LeetCode 701 / 700 / 98)

Insert and search walk down from the root (iteratively, so depth is never a problem). **Validate** is where people fail: comparing a node only with its children is **wrong**. In `[5, 4, 6, None, None, 3, 7]` every parent/child pair looks fine, but 3 sits in the right subtree of 5. The fix: pass down the allowed `(low, high)` range, or check that the in-order traversal is strictly increasing.

In [ ]:
def bst_insert(root, val):
    new = TreeNode(val)
    if root is None:
        return new
    node = root
    while True:
        side = "left" if val < node.val else "right"      # equal keys go right (a convention)
        child = getattr(node, side)
        if child is None:
            setattr(node, side, new)
            return root
        node = child

def bst_search(root, val):
    node = root
    while node and node.val != val:
        node = node.left if val < node.val else node.right
    return node

def is_valid_bst_naive(root):            # WRONG: only looks one level down
    if root is None:
        return True
    if (root.left and root.left.val >= root.val) or (root.right and root.right.val <= root.val):
        return False
    return is_valid_bst_naive(root.left) and is_valid_bst_naive(root.right)

def is_valid_bst(root, low=float("-inf"), high=float("inf")):
    if root is None:
        return True
    if not low < root.val < high:
        return False
    return is_valid_bst(root.left, low, root.val) and is_valid_bst(root.right, root.val, high)

trap = build_tree([5, 4, 6, None, None, 3, 7])
print("naive check says valid:", is_valid_bst_naive(trap), "| range check says valid:", is_valid_bst(trap))
assert is_valid_bst_naive(trap) and not is_valid_bst(trap)
draw_tree(trap, title="Every parent/child pair looks fine, but 3 < 5 sits right of 5",
          highlight={trap.right.left})

In [ ]:
root = None
for v in [8, 3, 10, 1, 6, 14, 4, 7, 13]:
    root = bst_insert(root, v)
assert to_level_list(root) == to_level_list(bst) and is_valid_bst(root)
assert bst_search(root, 6).val == 6 and bst_search(root, 5) is None
rnd = random.Random(0)
for _ in range(100):
    keys = rnd.sample(range(1000), rnd.randint(0, 50))
    r = None
    for k in keys:
        r = bst_insert(r, k)
    assert inorder_iter(r) == sorted(keys) and is_valid_bst(r)
print("100 random BSTs: in-order is sorted and the range check agrees")

**Why the height matters:** insert the same keys in sorted order and the "tree" is a linked list. Random order gives height ≈ a small multiple of `log2 n`. This is why production systems use self-balancing trees (red-black, AVL, B-trees in databases) — they guarantee O(log n) height.

In [ ]:
def height_iter(root):
    return len(level_order(root))

sizes = [10, 50, 100, 200, 400, 700, 1000]
sorted_h, random_h = [], []
for n in sizes:
    r = None
    for k in range(n):
        r = bst_insert(r, k)
    sorted_h.append(height_iter(r))
    trials = []
    for seed in range(5):
        keys = random.Random(seed).sample(range(n), n)
        r = None
        for k in keys:
            r = bst_insert(r, k)
        trials.append(height_iter(r))
    random_h.append(sum(trials) / len(trials))

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(sizes, sorted_h, color=BLUE, lw=2, marker="o", ms=4, label="keys inserted in sorted order")
ax.plot(sizes, random_h, color=ORANGE, lw=2, marker="o", ms=4, label="keys inserted in random order (mean of 5)")
ax.set_xlabel("number of keys n"); ax.set_ylabel("tree height"); ax.legend()
ax.set_title("BST height: sorted input degenerates to a list (h = n); random input stays ~log n")
plt.show()
assert sorted_h == sizes and max(random_h) < 40
print(f"n = 1000: sorted height {sorted_h[-1]}, random height {random_h[-1]:.1f}")

### Problem 6 — Lowest common ancestor (LeetCode 235 for BSTs, 236 for any binary tree)

The LCA of `p` and `q` is the deepest node that has both in its subtree (a node counts as its own descendant).

- **In a BST:** walk down from the root. If both values are smaller go left, if both are larger go right; otherwise this node is where their paths split — the answer. O(h), no recursion.
- **In any binary tree:** post-order. Return the node if it *is* `p` or `q`; if the left and right subtrees each report one, this node is the LCA; otherwise pass up whichever side found something. O(n).
- **Brute force:** compute the root-to-`p` and root-to-`q` paths and take the last node they share. Also O(n), but two passes and two path lists.

In [ ]:
def lca_bst(root, p, q):
    node = root
    while node:
        if p < node.val and q < node.val:
            node = node.left
        elif p > node.val and q > node.val:
            node = node.right
        else:
            return node

def lca(root, p, q):
    if root is None or root.val in (p, q):
        return root
    left, right = lca(root.left, p, q), lca(root.right, p, q)
    if left and right:
        return root                          # p and q are on different sides
    return left or right

def path_to(root, target):
    if root is None:
        return None
    if root.val == target:
        return [root]
    for child in (root.left, root.right):
        sub = path_to(child, target)
        if sub:
            return [root] + sub
    return None

def lca_by_paths(root, p, q):
    common = None
    for a, b in zip(path_to(root, p), path_to(root, q)):
        if a is not b:
            break
        common = a
    return common

tree236 = build_tree([3, 5, 1, 6, 2, 0, 8, None, None, 7, 4])
bst235 = build_tree([6, 2, 8, 0, 4, 7, 9, None, None, 3, 5])
assert [lca(tree236, *pq).val for pq in [(5, 1), (5, 4), (7, 8), (6, 4)]] == [3, 5, 3, 5]
assert [lca_bst(bst235, *pq).val for pq in [(2, 8), (2, 4), (3, 5)]] == [6, 2, 4]
values = inorder(bst235)
for p in values:
    for q in values:
        assert lca(bst235, p, q) is lca_bst(bst235, p, q) is lca_by_paths(bst235, p, q)
print("LCA: general, BST and path-based versions agree on every pair")

### Problem 7 — Serialize and deserialize (LeetCode 297)

Turn a tree into a string and back. Pre-order with a `#` marker for every missing child is the shortest correct answer: the markers say exactly where each subtree ends. Without them the shape is lost — `[1, 2]` (2 on the left) and `[1, None, 2]` (2 on the right) have the same pre-order `[1, 2]`.

In [ ]:
def serialize(root):
    out = []
    def walk(node):
        if node is None:
            out.append("#")
            return
        out.append(str(node.val)); walk(node.left); walk(node.right)
    walk(root)
    return ",".join(out)

def deserialize(data):
    tokens = iter(data.split(","))
    def build():
        tok = next(tokens)
        if tok == "#":
            return None
        node = TreeNode(int(tok))
        node.left = build()
        node.right = build()
        return node
    return build()

left_two, right_two = build_tree([1, 2]), build_tree([1, None, 2])
assert preorder(left_two) == preorder(right_two)               # same values, different shapes
assert serialize(left_two) != serialize(right_two)             # the markers keep them apart
print("serialize(example) =", serialize(example))
for t in [example, bst, None, build_tree([-1, -20, 30]), *(random_tree(40, s) for s in range(100))]:
    s = serialize(t)
    assert serialize(deserialize(s)) == s and to_level_list(deserialize(s)) == to_level_list(t)
print("round trip OK on the examples, the empty tree, negative values and 100 random trees")

**Complexity:** O(n) time and space both ways. For very deep trees swap the recursion for an explicit stack, or use the level-order format (`to_level_list` / `build_tree` above), which is iterative.

### Problem 8 — K-th smallest element in a BST (LeetCode 230)

In-order traversal of a BST is sorted, so the k-th value visited is the answer. **Brute force:** collect the whole in-order list and index it — O(n). **Optimal:** iterative in-order that **stops** after k pops — O(h + k).

In [ ]:
def kth_smallest(root, k):
    stack, node = [], root
    while stack or node:
        while node:
            stack.append(node); node = node.left
        node = stack.pop()
        k -= 1
        if k == 0:
            return node.val
        node = node.right
    raise ValueError("k is larger than the number of nodes")

assert kth_smallest(bst, 1) == 1 and kth_smallest(bst, 4) == 6
rnd = random.Random(1)
for _ in range(50):
    keys = rnd.sample(range(500), rnd.randint(1, 40))
    r = None
    for key in keys:
        r = bst_insert(r, key)
    assert [kth_smallest(r, k) for k in range(1, len(keys) + 1)] == sorted(keys)
try:
    kth_smallest(bst, 100)
except ValueError as e:
    print("k too large ->", e)
print("k-th smallest matches sorted(keys) for every k on 50 random BSTs")

**Follow-up they will ask:** "the BST changes often and you query k-th smallest a lot" → store the **subtree size** in every node (update it on insert/delete). Then at each node compare `k` with `size(left) + 1` and go left, stop, or go right with `k` reduced: O(h) per query.

## Try it yourself

**1. Path sum (LeetCode 112).** Is there a root-to-**leaf** path whose values add up to `target`? The empty tree has no path.

In [ ]:
# Solution — try it yourself first, then run this
def has_path_sum(root, target):
    if root is None:
        return False
    if root.left is None and root.right is None:      # a leaf: the path ends here
        return root.val == target
    rest = target - root.val
    return has_path_sum(root.left, rest) or has_path_sum(root.right, rest)

lc112 = build_tree([5, 4, 8, 11, None, 13, 4, 7, 2, None, None, None, 1])
assert all(has_path_sum(lc112, s) for s in (27, 22, 26, 18))    # its four root-to-leaf sums
assert not has_path_sum(lc112, 23)
assert not has_path_sum(build_tree([1, 2, 3]), 5) and not has_path_sum(None, 0)
assert not has_path_sum(build_tree([1, 2]), 1)           # 1 alone is not a root-to-leaf path
print("path sum 22 exists:", has_path_sum(lc112, 22))

**2. Right side view (LeetCode 199).** The values you would see standing to the right of the tree: the last node of every level.

In [ ]:
# Solution — try it yourself first, then run this
def right_side_view(root):
    return [level[-1] for level in level_order(root)]

assert right_side_view(build_tree([1, 2, 3, None, 5, None, 4])) == [1, 3, 4]
assert right_side_view(build_tree([1, 2, 3, 4])) == [1, 3, 4]
assert right_side_view(None) == []
print(right_side_view(build_tree([1, 2, 3, None, 5, None, 4])))

**3. AI-engineer flavour: breadcrumbs from a markdown outline.** A document's headings form a tree. Chunkers often prepend each chunk's heading path ("Guide > Install > Windows") so a retrieved chunk still says where it came from. Produce the root-to-heading path for every heading. Hint: keep a stack of the current ancestors; a heading of level L pops every stacked heading of level ≥ L.

In [ ]:
# Solution — try it yourself first, then run this
doc = """# Guide
intro text
## Install
### Windows
### macOS
## Usage
### CLI
#### Flags
## FAQ"""

def heading_breadcrumbs(markdown):
    crumbs, stack = [], []                      # stack: (level, title) of the current ancestors
    for line in markdown.splitlines():
        if not line.startswith("#"):
            continue
        level = len(line) - len(line.lstrip("#"))
        while stack and stack[-1][0] >= level:
            stack.pop()
        stack.append((level, line[level:].strip()))
        crumbs.append(" > ".join(title for _, title in stack))
    return crumbs

for crumb in heading_breadcrumbs(doc):
    print(crumb)
assert heading_breadcrumbs(doc)[2] == "Guide > Install > Windows"
assert heading_breadcrumbs(doc)[-1] == "Guide > FAQ" and heading_breadcrumbs("") == []

## Say it in an interview

**30-second answer:** "Most tree problems are either a DFS that returns something to the parent — height, a balanced flag, a best path — or a BFS that processes one level at a time. Both are O(n) time; DFS uses O(h) stack, which is O(log n) for a balanced tree and O(n) for a skewed one, so for deep trees I switch to an explicit stack. For a BST, in-order is sorted and search is O(h); the guarantee of O(log n) needs a self-balancing tree."

**Key facts:** n nodes → n − 1 edges · perfect tree of height h has 2^h − 1 nodes · random BST height is O(log n), sorted insertion gives height n · in-order of a BST is sorted.

**Follow-ups and traps:**
- *Validate BST:* pass `(low, high)` bounds down; checking only the children is wrong (the `[5, 4, 6, None, None, 3, 7]` tree).
- *Diameter:* counts edges, and the longest path may not pass through the root.
- *Balanced / diameter:* compute height and the answer in the **same** pass, or it becomes O(n²).
- *Serialize:* include null markers or the shape is ambiguous.
- *Iterative in-order:* "go left as far as possible, pop, go right" — practise until you can write it without thinking.
- *LCA:* the BST version uses the ordering (O(h)); the general version is a post-order that bubbles up "found p/q".

## Pattern cheat sheet

| Signal | Approach | Time / space |
|---|---|---|
| depth, height, balanced, diameter, path sums | DFS returning a value (post-order) | O(n) / O(h) |
| per level, right side view, minimum depth | BFS with `for _ in range(len(queue))` | O(n) / O(width) |
| sorted order of a BST, k-th smallest | in-order (iterative, stop early) | O(h + k) / O(h) |
| search / insert in a BST | walk down comparing keys | O(h) / O(1) |
| validate a BST | DFS with `(low, high)` bounds | O(n) / O(h) |
| LCA in a BST / in any tree | split point walking down / post-order bubbling up | O(h) / O(n) |
| copy or serialise | pre-order with null markers | O(n) / O(n) |
| very deep tree | explicit stack instead of recursion | same, no RecursionError |

## Next

- [10 · Heaps and top-k](10_heaps_and_top_k.ipynb) — a heap is a complete binary tree stored in an array.
- [11 · Graphs](11_graphs_bfs_dfs_topo_dijkstra.ipynb) — trees are graphs without cycles; BFS/DFS carry over.
- [14 · Tries, union-find and LRU cache](14_tries_union_find_lru_cache.ipynb) — a trie is an n-ary tree of characters.
- [08 · Recursion and backtracking](08_recursion_and_backtracking.ipynb) · [Chunking strategies](../09_rag/01_chunking_strategies_and_semantic_chunking.ipynb) (heading-aware chunks) · [Decision trees and boosting](../05_machine_learning/06_decision_trees_random_forest_boosting.ipynb)
- Theory: [DSA basics course — trees chapter](../../dsa_basics/index.html) · [interview bank](../../ai_interview_prep/index.html)